# Kombinierte Labels: Dimensionsreduktion auf Spektralbändern

**Hypothese:** Mindestens eine fold-gefittete Dimensionsreduktion bündelt die für die RBF-SVM relevanten Spektralunterschiede besser als die korrelierten Rohbänder.

**Versuchsplan:** PCA, ICA, NMF, OSP, LPP, VSRP und DBN werden jeweils mit 20, 40 und 80 Ausgabekomponenten verglichen. Jede Methode erhält die erforderliche Skalierung innerhalb der Pipeline; AEZ und Month umgehen die Reduktion. Die Methoden-, Komponenten- und SVM-Auswahl erfolgt ausschließlich über die fünf festen Trainingsfolds. Nur der Gesamtsieger wird einmal auf dem gemeinsamen Validierungssplit bewertet.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_combined_svm
from awp2.preprocessing import PreprocessingConfig

COMPONENT_COUNTS = (20, 40, 80)
LPP_NEIGHBORS = 10
DBN_HIDDEN_LAYER_WIDTHS = (64, 96, 128)
SVM_PARAM_GRID = {
    "estimator__C": [100.0, 300.0],
    "estimator__gamma": [0.001, 0.003],
}
REFERENCE_CV_BACC_COMBINED = 0.8473
REFERENCE_VALIDATION_BACC_COMBINED = 0.8664

reduction_candidates = {}
for component_count, hidden_layer_width in zip(
    COMPONENT_COUNTS, DBN_HIDDEN_LAYER_WIDTHS, strict=True
):
    reduction_candidates[f"pca_{component_count}"] = {
        "method": "PCA",
        "components": component_count,
        "preprocessing": PreprocessingConfig(
            use_meta=True,
            use_spectral_standard_scale=True,
            spectral_pca_components=component_count,
        ),
    }
    reduction_candidates[f"ica_{component_count}"] = {
        "method": "ICA",
        "components": component_count,
        "preprocessing": PreprocessingConfig(
            use_meta=True,
            use_spectral_standard_scale=True,
            spectral_ica_components=component_count,
        ),
    }
    reduction_candidates[f"nmf_{component_count}"] = {
        "method": "NMF",
        "components": component_count,
        "preprocessing": PreprocessingConfig(
            use_meta=True,
            spectral_nmf_components=component_count,
        ),
    }
    reduction_candidates[f"osp_{component_count}"] = {
        "method": "OSP",
        "components": component_count,
        "preprocessing": PreprocessingConfig(
            use_meta=True,
            spectral_osp_components=component_count,
        ),
    }
    reduction_candidates[f"lpp_{component_count}"] = {
        "method": "LPP",
        "components": component_count,
        "preprocessing": PreprocessingConfig(
            use_meta=True,
            use_spectral_standard_scale=True,
            spectral_lpp_components=component_count,
            spectral_lpp_neighbors=LPP_NEIGHBORS,
        ),
    }
    reduction_candidates[f"vsrp_{component_count}"] = {
        "method": "VSRP",
        "components": component_count,
        "preprocessing": PreprocessingConfig(
            use_meta=True,
            spectral_vsrp_components=component_count,
        ),
    }
    reduction_candidates[f"dbn_{component_count}"] = {
        "method": "DBN",
        "components": component_count,
        "preprocessing": PreprocessingConfig(
            use_meta=True,
            use_spectral_minmax_scale=True,
            spectral_dbn_layers=(hidden_layer_width, component_count),
        ),
    }

pd.DataFrame(
    {
        "Methode": candidate["method"],
        "Komponenten": candidate["components"],
        "Vorverarbeitung": candidate["preprocessing"].model_dump(exclude_defaults=True),
    }
    for candidate in reduction_candidates.values()
)

## 1. Dimensionsreduktion und SVM mit Cross-Validation vergleichen

Die 21 Kandidaten kombinieren sieben offiziell implementierte Reduktionsverfahren mit 20, 40 und 80 Ausgabekomponenten. Jede Pipeline wird in jedem Fold neu gefittet. Pro Kandidat sucht dieselbe kleine RBF-SVM-Umgebung nach `C` und `gamma`; der gemeinsame Validierungssplit bleibt während der Auswahl unberührt.

In [ ]:
tuned_by_candidate = {}

for candidate_name, candidate in reduction_candidates.items():
    tuned_by_candidate[candidate_name] = tune(
        make_combined_svm(),
        TuneConfig(
            name=f"combined_svm_reduction_{candidate_name}_search",
            approach="combined",
            description=(
                f"RBF-SVM auf {candidate['method']} mit {candidate['components']} "
                "reduzierten Spektralkomponenten; AEZ und Month bleiben getrennte Merkmale."
            ),
            preprocessing=candidate["preprocessing"],
            param_grid=SVM_PARAM_GRID,
        ),
    )

candidate_summary = (
    pd.DataFrame(
        [
            {
                "Kandidat": candidate_name,
                "Methode": candidate["method"],
                "Komponenten": candidate["components"],
                "CV BAcc kombiniert": tuned_by_candidate[candidate_name].best_score,
                "Beste Parameter": tuned_by_candidate[candidate_name].best_params,
            }
            for candidate_name, candidate in reduction_candidates.items()
        ]
    )
    .sort_values("CV BAcc kombiniert", ascending=False)
    .reset_index(drop=True)
)
display(candidate_summary)

## 2. Den globalen CV-Gewinner einmal validieren

Die Auswahl berücksichtigt ausschließlich die vollständige CV-Rangfolge aller Reduktionsverfahren. Die Validierung beantwortet danach nur noch, ob der ausgewählte Kandidat die Rohspektren-Referenz auf dem gemeinsamen Holdout übertrifft.

In [ ]:
best_candidate_name = candidate_summary.iloc[0]["Kandidat"]
best_candidate = reduction_candidates[best_candidate_name]
best_tuned = tuned_by_candidate[best_candidate_name]

result = run(
    best_tuned.best_model,
    RunConfig(
        name="combined_svm_spectral_reduction",
        approach="combined",
        description=(
            "Beste RBF-SVM aus dem CV-Screening von sieben Spektralreduktionen, einmal auf "
            "dem gemeinsamen Validierungssplit bewertet."
        ),
        preprocessing=best_candidate["preprocessing"],
        tuning_run=best_tuned.run_id,
    ),
)

{
    "Kandidat": best_candidate_name,
    "Methode": best_candidate["method"],
    "Komponenten": best_candidate["components"],
    "CV BAcc kombiniert": best_tuned.best_score,
    "CV-Parameter": best_tuned.best_params,
    "Validierungsmetriken": result.metrics.model_dump(),
    "Tuning-Run": best_tuned.run_id,
    "Bewertungs-Run": result.run_id,
}

## 3. Ergebnis prüfen

Die Analyse stellt den globalen CV-Gewinner der sieben Reduktionsverfahren der aktuellen Rohspektren-Referenz gegenüber. Alle Vorhersagen müssen weiterhin gültige Crop/Stage-Paare sein.

In [ ]:
from awp2.evaluation import plot_confusion_matrices
from awp2.plots import (
    plot_class_recall,
    plot_metric_comparison,
    plot_top_confusions,
    plot_tuning_candidates,
)

reference_metrics = result.metrics.model_copy(
    update={
        "bacc_crop": 0.931,
        "bacc_stage": 0.906,
        "bacc_combined": REFERENCE_VALIDATION_BACC_COMBINED,
    }
)
metric_ax = plot_metric_comparison(
    {
        "Rohspektren-Referenz": reference_metrics,
        f"{best_candidate['method']} ({best_candidate['components']})": result.metrics,
    },
)
metric_ax.set_title("Kombinierte Labels: Dimensionsreduktion gegen Rohspektren")
metric_ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
metric_ax.figure.subplots_adjust(bottom=0.22)
display(metric_ax.figure)

tuning_ax = plot_tuning_candidates(
    best_tuned.candidates,
    color_by="estimator__C",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title(
    f"RBF-SVM mit {best_candidate['method']} und {best_candidate['components']} Komponenten"
)
display(tuning_ax.figure)

for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

assert result.metrics.invalid_combinations == 0.0

In [ ]:
from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import save_doc_figure

save_doc_figure(
    metric_ax.figure,
    "combined_svm_spectral_reduction_comparison",
    MODEL_DOCS_FIGURES_DIR,
)
save_doc_figure(
    tuning_ax.figure,
    "combined_svm_spectral_reduction_tuning",
    MODEL_DOCS_FIGURES_DIR,
)
save_doc_figure(
    plot_confusion_matrices(result.y_val, result.y_pred),
    "combined_svm_spectral_reduction_confusion",
    MODEL_DOCS_FIGURES_DIR,
)
save_doc_figure(
    plot_class_recall(result.y_val, result.y_pred),
    "combined_svm_spectral_reduction_recall",
    MODEL_DOCS_FIGURES_DIR,
)
save_doc_figure(
    plot_top_confusions(result.y_val, result.y_pred),
    "combined_svm_spectral_reduction_confusions",
    MODEL_DOCS_FIGURES_DIR,
)

## Fazit

PCA mit 20 Komponenten gewinnt das vollständige CV-Screening der sieben Reduktionsverfahren mit einer kombinierten BAcc von 0.8531. Ihr einmaliges Validierungsergebnis von 0.8634 liegt dennoch 0.0030 unter der Rohspektren-Referenz von 0.8664. Die Hypothese wird daher nicht gestützt; `combined_svm_expanded` bleibt die Referenz.